## 💰 資金配置與避險原則 (Capital Allocation & Hedging)

本回測框架中，所有配對交易策略均嚴格遵循以下資金配置與避險權重原則，這也是計算 **RCC** 與 **REC** 的基礎學術底層：

1. **等資金多組合分配 (Portfolio Equal-Weighting)**：
   - 設定投資組合總初始虛擬本金為 $C_{Total} = \$10,000$。
   - 根據策略篩選出的最優配對數 $N_{pair}$ (即 `Top N`)，將資金等權重分配至各交易組，單一組的分配金額限制為：
     $$C_{pair} = \frac{C_{Total}}{N_{pair}}$$
2. **避險比例 (Hedge Ratio) 與多空市值分配**：
   - 對於每一對 Pair A (因變數) 與 Pair B (自變數)，以迴歸係數 $\beta_t$ (Hedge Ratio) 進行市值對沖配置：
     $$Weight_A = \frac{1.0}{1.0 + |\beta_t|}, \quad Weight_B = \frac{|\beta_t|}{1.0 + |\beta_t|}$$
   - **美元中性避險**：當 Z-Score 觸發進場訊號時，做多 A 市值 $C_{pair} \cdot Weight_A$，同時做空 B 市值 $C_{pair} \cdot Weight_B$ (或反向配置)，保證組合的 Beta 中性與初始美元中性。


## 📌 策略概覽與對照表

::: {.smaller}

| 策略檔案 | 策略名稱 | 形成期篩選核心機制 | 避險比例 (Hedge Ratio) 計算 | 交易期核心特色 | 學術依據與 `ref/` 對應文獻 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **[ssd_basic.py](file:///d:/Unknown/Papper/Code/strategies/ssd_basic.py)** | **經典 SSD 距離法** | 累積總回報指數的平方歐氏距離 (SSD) | 固定為 1.0 (等額配置) | 傳統 Z-Score 突破交易 | Gatev et al. (2006)<br>`2006-Pairs Trading Performance...pdf` |
| **[ssd.py](file:///d:/Unknown/Papper/Code/strategies/ssd.py)** | **進階 SSD 與 OLS 斜率法** | Z-Score 標準化對數價格的 SSD | 靜態/滾動 OLS 斜率 $\beta$ | OLS 避險市值中性配置<br>OLS 殘差滾動 Z-Score | OLS 殘差理論與統計套利綜述<br>`2006-A New Approach to Modeling...pdf`<br>`2015-Statistical arbitrage pairs...pdf` |
| **[eg.py](file:///d:/Unknown/Papper/Code/strategies/eg.py)** | **Engle-Granger 共整合法** | 相關性快篩 + 雙向 EG 共整合檢定 (ADF $p < 0.01$) | 靜態 OLS 共整合係數 $\beta$ | **Optimal Double Stopping** 訊號<br>進場防假突破機制<br>停損永久冷凍 | Engle and Granger (1987)<br>`2014-Pairs trading and selection...pdf`<br>`2015-Statistical arbitrage pairs...pdf` |
| **[HDBSCAN.py](file:///d:/Unknown/Papper/Code/strategies/HDBSCAN.py)** | **HDBSCAN 多維特徵分群** | 13維特徵 + UMAP 降維 + HDBSCAN 分群<br>同群落 × 同產業 EG 共整合 + OU 半衰期過濾 | 靜態/滾動 OLS 共整合係數 $\beta$ | 波動度動態調整 Z-Score<br>**投資組合總體止損斷路器** | 無監督學習分群配對與 OU 過程<br>`2021-Pairs Trading via Unsupervised...pdf`<br>`2008-Optimal Pairs Trading...pdf` |
| **[HDBSCAN_Autoencoder.py](file:///d:/Unknown/Papper/Code/strategies/HDBSCAN_Autoencoder.py)** | **Autoencoder 深度表徵分群** | 收益率序列 + PyTorch Autoencoder 深度特徵<br>UMAP 降維 + HDBSCAN 分群<br>同群落 × 同產業 EG 共整合 + OU 半衰期過濾 | 靜態/滾動 OLS 共整合係數 $\beta$ | 波動度動態調整 Z-Score<br>**投資組合總體止損斷路器** | 深度自編碼器與機器學習配對<br>`2021-Pairs Trading via Unsupervised...pdf`<br>`2018-配對交易與機器學習在台灣...pdf`<br>`2021-透過機器學習及標記技術...pdf` |
| **[HDBSCAN_MultiFactor.py](file:///d:/Unknown/Papper/Code/strategies/HDBSCAN_MultiFactor.py)** | **時序多因子特徵分群** | 6大金融因子空間 + 直接 HDBSCAN 分群<br>同群落 × 同產業 EG 共整合 + OU 半衰期過濾 | 靜態/滾動 OLS 共整合係數 $\beta$ | 波動度動態調整 Z-Score<br>**投資組合總體止損斷路器** | 金融因子空間配對<br>`2021-In Search of Pairs using Firm...pdf`<br>CAPM 模型 (Sharpe 1964) |

:::


# 1️⃣ 經典 SSD

## 1️⃣ 經典 SSD：Formation 形成期

經典 SSD 策略是配對交易最為經典的「距離法」實現。其核心思想是尋找在歷史形成期中價格走勢最為貼合的股票對。步驟如下：

1. **價格正規化**：將形成期內的股價序列 $P_{i, t}$ 轉換為**累積總回報指數** $P_{i, t}^{norm}$。將形成期首日價格歸一化為 $1.0$：
   $$P_{i, t}^{norm} = \frac{P_{i, t}}{P_{i, 0}}$$
2. **計算平方差之和 (SSD)**：在同一個 GICS 產業分類中，計算兩兩股票 $A$ 與 $B$ 在歸一化價格序列上的平方歐氏距離：
   $$SSD_{A, B} = \sum_{t=1}^{T_{Form}} \left( P_{A, t}^{norm} - P_{B, t}^{norm} \right)^2$$
3. **配對選擇**：將產業內所有可能配對的 $SSD$ 按升序排序，選擇最小的前 $N$ 組配對（`top_n`）作為交易標的。在距離法中，避險比例固定為 $\beta = 1.0$，即等權重美元中性配置（Dollar Weighting）。


## 1️⃣ 經典 SSD：Trading 交易期

1. **計算價差與 Z-Score**：
   - **無滾動均值去中心化 (`zscore_window == 0`)**：以形成期計算出的價差均值 $\mu_{Spread, Form}$ 與標準差 $\sigma_{Spread, Form}$ 來計算交易期的 $Z_t$：
     $$Spread_t = P_{A, t}^{norm} - P_{B, t}^{norm}$$
     $$Z_t = \frac{Spread_t - \mu_{Spread, Form}}{\sigma_{Spread, Form}}$$
   - **滾動去中心化 (`zscore_window > 0`)**：採用交易期滾動視窗 $W$ 的均值與標準差，避險比例鎖定為 $1.0$：
     $$Spread_t = P_{A, t}^{norm} - SMA\left(P_A^{norm} - P_B^{norm}, W\right)_t - P_{B, t}^{norm}$$
     $$Z_t = \frac{Spread_t}{\sigma_{Spread, W, t}}$$
2. **交易訊號與停損**：
   - **做空價差**：當 $Z_t > EntryZ$ 且配對不在冷卻期時，進場：$Position = -1$。
   - **做多價差**：當 $Z_t < -EntryZ$ 且配對不在冷卻期時，進場：$Position = 1$。
   - **平倉**：多頭當 $Z_t \ge -ExitZ$ 平倉；空頭當 $Z_t \le ExitZ$ 平倉。
   - **停損與冷卻**：單次交易虧損比例達 `stop_loss_pct` 強制平倉。若 `allow_reentry = False` 永久冷凍；若 `allow_reentry = True` 則記錄冷卻方向 `cooldown_dir` 直到 $Z$ 回歸零軸才解除。


## 1️⃣ 經典 SSD：學術理論與 ref/ 依據

- **文獻依據**：**Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006)**. *"Pairs Trading: Performance of a Relative-Value Arbitrage Rule"*。
- **對應 ref/ 內檔案**：[`2006-Pairs Trading Performance of a Relative-Value Arbitrage Rule.pdf`](file:///d:/Unknown/Papper/Code/ref/2006-Pairs%20Trading%20Performance%20of%20a%20Relative-Value%20Arbitrage%20Rule.pdf)。
- **依據說明**：此策略完全對應 Gatev et al. (2006) 論文中的第一天歸一化（Cumulative Total Returns Index normalized to 1.0 on the first day of the formation period）以及最小平方差總和（SSD）配對選擇法。論文中使用 2.0 倍標準差作為進場門檻，並在價差回歸均值時平倉，此公式完全體現在程式碼的 `Formation` 與 `Trading` 類中。


# 2️⃣ 進階 SSD

## 2️⃣ 進階 SSD：Formation 形成期

相較於傳統等權重 SSD 策略，進階 SSD 策略引入了**對數價格變換**與 **OLS 迴歸係數**，以實現市值非對稱的中性避險配置。步驟如下：

1. **對數轉換與 Z-Score 標準化**：首先將形成期的價格序列 $P_{i, t}$ 轉換為對數價格，並進行 Z-Score 標準化（去中心化與方差歸一）：
   $$P_{i, t}^{norm} = \frac{\ln(P_{i, t}) - \mu_{\ln(P_i)}}{\sigma_{\ln(P_i)}}$$
   入其 $\mu_{\ln(P_i)}$ 與 $\sigma_{\ln(P_i)}$ 為形成期對數價格的均值與標準差。
2. **計算 SSD**：計算標準化後對數價格的平方歐氏距離 $SSD_{A, B}$。
3. **計算 Hedge Ratio (OLS 斜率 $\beta$)**：對於每個可能配對，以 $B$ 為自變數（自變數 $X$），$A$ 為因變數（因變數 $Y$），執行普通最小二乘法 (OLS) 迴歸：
   $$P_{A, t}^{norm} = \alpha + \beta P_{B, t}^{norm} + \epsilon_t$$
   $$\beta = \frac{Cov\left(P_A^{norm}, P_B^{norm}\right)}{Var\left(P_B^{norm}\right)}$$
   $$Spread_t = P_{A, t}^{norm} - \beta P_{B, t}^{norm}$$
   計算 OLS 殘差 $\epsilon_t$（即 $Spread$）在形成期的均值 $\mu_{Spread, Form}$ 與標準差 $\sigma_{Spread, Form}$。


## 2️⃣ 進階 SSD：Trading 交易期

1. **資金與部位分配**：根據估計出的 $\beta$（Hedge Ratio）進行市值分配：
   $$Weight_A = \frac{1.0}{1.0 + |\beta|}, \quad Weight_B = \frac{|\beta|}{1.0 + |\beta|}$$
2. **Z-Score 計算**：
   - **固定視窗 (`zscore_window == 0`)**：使用 Formation 期的 Spread 均值與標準差進行正規化。
   - **滾動視窗 (`zscore_window > 0`)**：在交易期每個滾動視窗 $W$ 內重新計算 OLS 斜率 $\beta_t$、截距 $\alpha_t$ 與價差：
     $$\beta_t = \frac{Cov\left(P_A^{norm}, P_B^{norm}\right)_W}{Var\left(P_B^{norm}\right)_W}$$
     $$\alpha_t = SMA(P_A^{norm}, W)_t - \beta_t SMA(P_B^{norm}, W)_t$$
     $$Spread_t = P_{A, t}^{norm} - \alpha_t - \beta_t P_{B, t}^{norm}$$
     根據 OLS 理論，殘差標準差可利用協方差矩陣直接推導計算以提升運算效率：
     $$\sigma_{\epsilon, t} = \sqrt{Var\left(P_A^{norm}\right)_W - \beta_t Cov\left(P_A^{norm}, P_B^{norm}\right)_W} = \sqrt{\sigma_{A, W, t}^2 - \beta_t \sigma_{AB, W, t}}$$
     $$Z_t = \frac{Spread_t}{\max\left(\sigma_{\epsilon, t}, min\_spread\_std\right)}$$
3. **交易訊號**：交易邏輯與 `ssd_basic.py` 相同，但進場時的資金分配完全基於時變的避險比例 $\beta_t$ 權重配置。


## 2️⃣ 進階 SSD：學術理論與 ref/ 依據

- **文獻依據**：
  1. **Do, B., & Faff, R. (2010)**. *"Does Simple Pairs Trading Still Work?"*（對應 [`2018-Does Simple Pairs Trading Still Work.pdf`](file:///d:/Unknown/Papper/Code/ref/2018-Does%20Simple%20Pairs%20Trading%20Still%20Work.pdf)）。
  2. **Krauss, C. (2017)**. *"Statistical arbitrage pairs trading strategies: Review and outlook"*（對應 [`2015-Statistical arbitrage pairs trading strategies Review and outlook.pdf`](file:///d:/Unknown/Papper/Code/ref/2015-Statistical%20arbitrage%20pairs%20trading%20strategies%20Review%20and%20outlook.pdf)）。
  3. **Elliott et al. (2005)**. *"A New Approach to Modeling and Estimation for Pairs Trading"*（對應 [`2006-A New Approach to Modeling and Estimation for Pairs Trading.pdf`](file:///d:/Unknown/Papper/Code/ref/2006-A%20New%20Approach%20to%20Modeling%20and%20Estimation%20for%20Pairs%20Trading.pdf)）。
- **學術邏輯**：相較於傳統距離法，將資產價格取對數並通過 OLS 估計 $\beta$ 是克服資產非平穩性與實現「貝塔中性」的標準做法。程式中 $Var(Y - eta X) = Var(Y) - eta Cov(X,Y)$ 的快速殘差標準差估計，是最小二乘迴歸的標準統計學公式，有效避免了生成殘差數組再求標準差的二次運算開銷。


# 3️⃣ Engle-Granger 共整合

## 3️⃣ Engle-Granger 共整合：Formation 形成期

Engle-Granger 共整合策略是一種基於**嚴謹計量經濟學理論**的配對選擇方法。它要求配對股票的對數價格之間具有穩健的長期均衡關係。步驟如下：

1. **Pearson 相關係數快篩**：僅保留相關性 $\rho \ge 0.7$ 的配對，保證同向走勢：
   $$\rho = \frac{Cov\left(\ln(P_A), \ln(P_B)\right)}{\sigma_{\ln(P_A)} \sigma_{\ln(P_B)}}$$
2. **雙向 OLS 迴歸與 ADF 殘差定態檢定**：
   - 進行雙向迴歸，計算避險比例（OLS 斜率 $\beta$）與截距 $\alpha$：
     $$\ln(P_{A, t}) = \alpha + \beta \ln(P_{B, t}) + \epsilon_t$$
     $$\beta = \frac{Cov\left(\ln(P_A), \ln(P_B)\right)}{Var\left(\ln(P_B)\right)}, \quad Spread_t = \ln(P_{A, t}) - \alpha - \beta \ln(P_{B, t})$$
   - 對殘差序列 $\epsilon_t$（即 Spread）進行無截距、無趨勢項的**增強迪基-福勒 (ADF) 檢定**：
     $$\Delta \epsilon_t = \gamma \epsilon_{t-1} + \sum_{i=1}^{p} \varphi_i \Delta \epsilon_{t-i} + u_t$$
     檢定虛無假設 $H_0: \gamma = 0$ (存在單根，即無共整合)。若 $p\text{-value} < 0.01$，則確立共整合關係。
3. **方向篩選與排序**：選擇 $p\text{-value}$ 較小的方向為主迴歸，要求 $\beta > 0$，按 ADF 統計量由小到大排序選前 $N$ 組配對。


## 3️⃣ Engle-Granger 共整合：Trading 交易期

1. **Optimal Double Stopping 進場濾鏡**：為了防範殘差序列的「假突破」與順向趨勢漫步風險，本策略採用了**觸發後反向穿越建倉**的二次確認機制：
   - **空手且未觸發狀態 (`armed == 0`)**：
     - 若 $Z_t > EntryZ$ (預設 2.0)：觸發上軌，設置 `armed = -1` (準備做空)。
     - 若 $Z_t < -EntryZ$ (預設 -2.0)：觸發下軌，設置 `armed = 1` (準備做多)。
   - **已觸發上軌 (`armed == -1`)**：
     - 若 $Z_t$ 向下**回落穿越門檻**至 $ExitZ < Z_t < EntryZ$ 時，正式進場做空 A 做多 B：$Position = -1$。
     - 若 $Z_t$ 直接回歸到零軸下方 ($Z_t \le ExitZ$)，則重置 `armed = 0`（不進場）。
   - **已觸發下軌 (`armed == 1`)**：
     - 若 $Z_t$ 向上**回升穿越門檻**至 $-EntryZ < Z_t < ExitZ$ 時，正式進場做多 A 做空 B：$Position = 1$。
     - 若 $Z_t$ 直接回升到零軸上方 ($Z_t \ge ExitZ$)，則重置 `armed = 0`。
2. **部位配置與平倉**：部位按 $\beta$ 避險比例分配。當 Z-Score 回歸到平倉帶門檻（$ExitZ \pm ExitBuffer$）時平倉。
3. **停損永久冷凍**：當虧損達到停損門檻時強制平倉，並將 `is_stopped` 設為 `True`，**本回測期內該配對永久停交易**。因為共整合關係破裂代表長期均衡失效，不應重入。


## 3️⃣ Engle-Granger 共整合：學術理論與 ref/ 依據

- **文獻依據**：
  1. **Engle, R. F., & Granger, C. W. (1987)**. *"Co-integration and Error Correction: Representation, Estimation, and Testing"*。
  2. **Rad, H., Low, R. K. Y., & Faff, R. (2016)**. *"The profitability of pairs trading strategies: distance, cointegration, and copula methods"*（對應 [`2016-The profitability of pairs trading strategies distance, cointegration, and copula methols.pdf`](file:///d:/Unknown/Papper/Code/ref/2016-The%20profitability%20of%20pairs%20trading%20strategies%20distance,%20cointegration,%20and%20copula%20methols.pdf)）。
  3. **Puspaningrum et al. (2013)**. *"Evaluation of pairs-trading strategy..."*（對應 [`2009-Evaluation of pairs-trading strategy at the Brazilian financial market.pdf`](file:///d:/Unknown/Papper/Code/ref/2009-Evaluation%20of%20pairs-trading%20strategy%20at%20the%20Brazilian%20financial%20market.pdf)）。
- **學術邏輯**：此策略基於計量經濟學著名的 Engle-Granger 兩步共整合法。ADF 檢定不含常數與趨勢項，完美貼合殘差定態檢定的規範。Optimal Double Stopping 避險與建倉邏輯，來源於量化金融中解決 Ornstein-Uhlenbeck 過程交易時的最優停止時間問題，能極大地過濾市場在危機或重組時期的假突破訊號。


# 4️⃣ HDBSCAN 特徵分群

## 4️⃣ HDBSCAN 特徵分群：Formation 形成期

此策略將無監督學習引入配對交易，利用股票歷史走勢的 13 維金融統計與動力學特徵，進行高維空間密度分群，篩選出物理特徵最為貼近的 Pairs。

1. **13維金融統計特徵工程**：
   - **動量因子 (4維)**：$R_{Momentum, W} = \ln(P_t) - \ln(P_{t-W})$ ($W \in \{5, 21, 63, 126\}$)。
   - **波動度因子 (3維)**：21, 63 日及全期收益率標準差 $\sigma_W$。
   - **自相關因子 (3維)**：一階、五階、廿一階自相關係數 $\rho_k = \frac{Cov(R_t, R_{t-k})}{Var(R_t)}$。
   - **統計矩因子 (2維)**：收益率偏度 (Skewness) $S$ 與峰度 (Kurtosis) $K$。
   - **混沌動力學因子 (1維)**：**Hurst 指數** $H$ 衡量時間序列持久性（若 $H > 0.5$ 具趨勢持久性，若 $H < 0.5$ 具均值復歸性）。藉由重標極差分析（R/S Analysis）：
     $$\left(\frac{R}{S}\right)_k = C \cdot k^H \implies \ln\left(\frac{R}{S}\right)_k = H \ln(k) + \ln(C)$$
2. **降維與分群**：經 Z-Score 標準化後，利用 **UMAP** 將特徵空間降至 5 維，再執行 **HDBSCAN** 密度聚類。
3. **均值復歸半衰期 (Half-life) 過濾**：通過 EG 檢定後，將價差建模為 **Ornstein-Uhlenbeck 過程**：
   $$d\epsilon_t = -\theta \epsilon_t dt + \sigma dW_t$$
   離散化為 AR(1) 迴歸模型：$\Delta \epsilon_t = \alpha_1 + \lambda \epsilon_{t-1} + e_t$。若 $\lambda < 0$，則半衰期以 $T_{1/2} = -\frac{\ln(2)}{\lambda}$ 計算，限制 $2.0 \le T_{1/2} \le 60.0$ 天。


## 4️⃣ HDBSCAN 特徵分群：Trading 交易期

本策略在 OLS 價差 Z-Score 基礎上，引入了**動態波動度調整**與**投資組合總體止損斷路器**：

1. **時變波動度調整因子 (Volatility Adjustment)**：為了防範交易期市場波動率突增導致 Z-Score 異常鈍化或劇烈波動，以滾動 20 天的價差標準差對形成期標準差進行動態放大：
   $$vol\_factor_t = \max\left(1.0, \frac{\sigma_{Spread, 20, t}}{\sigma_{Spread, Form}}\right)$$
   $$adjusted\_std_t = \max\left(\sigma_{Spread, Form} \cdot vol\_factor_t, min\_spread\_std\right)$$
   $$Z_t = \frac{Spread_t - \mu_{Spread, Form}}{adjusted\_std_t}$$
2. **後置投資組合總體止損斷路器 (Portfolio Circuit Breaker)**：在每日交易結束後，計算本期投資組合所有配對的總累計損益。一旦總虧損達到總分配資金的 `portfolio_stop_loss_pct`（如 10%），**立即觸發全組合強制平倉，本期剩餘天數內全部配對永久停交易**，鎖定最大尾部風險。


## 4️⃣ HDBSCAN 特徵分群：學術理論與 ref/ 依據

- **文獻依據**：
  1. **Hanoch, Y., & Levy, H. (2021)**. *"Pairs Trading via Unsupervised Learning"*（對應 [`2021-Pairs Trading via Unsupervised Learning.pdf`](file:///d:/Unknown/Papper/Code/ref/2021-Pairs%20Trading%20via%20Unsupervised%20Learning.pdf)）。
  2. **Stochastic Control & Half-life**：參考 [`2008-Optimal Pairs Trading A Stochastic Control Approach.pdf`](file:///d:/Unknown/Papper/Code/ref/2008-Optimal%20Pairs%20Trading%20A%20Stochastic%20Control%20Approach.pdf)。
- **學術邏輯**：`2021-Pairs Trading via Unsupervised Learning.pdf` 系統性地闡述了利用高維統計特徵對股票進行特徵描述，並結合 UMAP 與密度聚類演算法（如 HDBSCAN）進行分群的完整框架。Ornstein-Uhlenbeck (O-U) 均值復歸半衰期公式 $T_{1/2} = -\ln(2)/\lambda$ 則是計量金融中衡量價差偏離均值後拉回速度的經典學術標準測度。


# 5️⃣ Autoencoder 分群

## 5️⃣ Autoencoder 分群：Formation 形成期

與手動特徵工程不同，深度表徵策略直接將歷史形成期的**日收益率時間序列**視為極高維的特徵，利用深度學習中的**去噪自編碼器 (Denoising Autoencoder)** 進行非線性無監督表徵學習，自動壓縮並擷取深層潛在特徵。

1. **構建 PyTorch MLP 自編碼器**：
   - **輸入維度**：$$d = T - 1$$ (其中 $d$ 為日收益率特徵數，已徹底消滅行內底線重疊 Bug)。
   - **Denoising Encoder 潛在空間壓縮**：
     $$h = \tanh\left( W_1 X_i + b_1 \right)$$
     $$h_{drop} = \text{Dropout}\left( h, 0.3 \right)$$
     $$z = W_2 h_{drop} + b_2 \quad (z \in \mathbb{R}^8)$$
     將高維序列壓縮至 8 維深度潛在空間。藉由 Dropout 隨機遮蔽 30% 特徵強制網路學習股票共同波動的低維特徵。
   - **Symmetric Decoder 重構**：
     $$\hat{h} = \tanh\left( W_3 z + b_3 \right)$$
     $$\hat{h}_{drop} = \text{Dropout}\left( \hat{h}, 0.3 \right)$$
     $$\hat{X}_i = W_4 \hat{h}_{drop} + b_4$$
2. **損失函數與優化**：利用 Adam 最小化 MSE 重構誤差並引入 L2 正則化（Weight Decay $10^{-4}$）：
   $$\mathcal{L}_{MSE} = \frac{1}{N}\sum_{i=1}^N \|X_i - \hat{X}_i\|^2 + \lambda \sum_{l=1}^4 \|W_l\|_F^2$$
   提取 8 維特徵 $z$ 作為降維與 HDBSCAN 分群的依據。


## 5️⃣ Autoencoder 分群：Trading 交易期

交易期算法與 `HDBSCAN.py` 相同：

- **部位分配**：雙端等市值中性配置。
- **波動度調整**：基於 20 日價差滾動標準差的動態 Z-Score 公式：
  $$vol\_factor_t = \max\left(1.0, \frac{\sigma_{Spread, 20, t}}{\sigma_{Spread, Form}}\right)$$
  $$adjusted\_std_t = \max\left(\sigma_{Spread, Form} \cdot vol\_factor_t, min\_spread\_std\right)$$
  $$Z_t = \frac{Spread_t - \mu_{Spread, Form}}{adjusted\_std_t}$$
- **投資組合總體止損斷路器**：當本期投資組合總虧損達到本金的 `portfolio_stop_loss_pct`（如 10%）時，強制將所有部位平倉，且本期剩餘天數內全部配對永久停交易，防範極端系統性風險。


## 5️⃣ Autoencoder 分群：學術理論與 ref/ 依據

- **文獻依據**：
  1. **Denoising Autoencoders**：Vincent, P. et al. (2008). *"Extracting and composing robust features with denoising autoencoders"*。
  2. **機器學習與深度學習在台灣市場實證**：
     - [`2018-配對交易與機器學習在台灣股票市場之應用.pdf`](file:///d:/Unknown/Papper/Code/ref/2018-配對交易與機器學習在台灣股票市場之應用.pdf)
     - [`2021-透過機器學習及標記技術建構配對交易策略.pdf`](file:///d:/Unknown/Papper/Code/ref/2021-透過機器學習及標記技術建構配對交易策略.pdf)
  3. **Autoencoder for Feature Compression**：參考 [`2021-Pairs Trading via Unsupervised Learning.pdf`](file:///d:/Unknown/Papper/Code/ref/2021-Pairs%20Trading%20via%20Unsupervised%20Learning.pdf)。
- **學術邏輯**：在量化金融中，收益率序列維度極高且含有大量噪聲。去噪自編碼器（DAE）通過隨機遮蔽強制網路學習股票共同波動的低維本質表徵。台灣實證研究論文詳細展示了如何利用 Autoencoder 提取特徵，並結合共整合法以顯著提升台灣市場配對交易的超額回報。


# 6️⃣ 多因子分群

## 6️⃣ 多因子分群：Formation 形成期

本策略捨棄了非線性的降維步驟（跳過 UMAP），直接在** 6 大經典金融時序多因子特徵空間**中進行 HDBSCAN 密度分群。這 6 大因子是學術界與業界公認刻畫個股系統性風險與特質風險的最核心指標：

1. **6大金融多因子時序特徵**：
   - **市場貝塔 (Market Beta, $\beta_{Market}$)**：對市場整體回報的系統性風險敏感度（$R_m$ 為市場成分股日均回報）：
     $$\beta_{Market} = \frac{Cov\left(R_i, R_m\right)}{Var\left(R_m\right)}$$
   - **總波動率 (Total Volatility, $\sigma_i$)**：日回報率標準差，反映總風險。
   - **收益偏度 (Skewness)**：日回報偏度，刻畫收益分佈的不對稱性。
   - **收益峰度 (Kurtosis)**：日回報超額峰度，反映極端事件的肥尾風險。
   - **趨勢斜率 (Trend Slope, $\gamma_{Trend}$)**：將對數價格對時間 $t$ 進行 OLS 迴歸得到長期趨勢斜率：$\ln(P_{i, t}) = \alpha_{Trend} + \gamma_{Trend} \cdot t + u_t$。
   - **特異波動率 (Idiosyncratic Volatility, $\sigma_{Idio}$)**：CAPM 迴歸殘差的標準差，反映個股特質風險：
     $$R_{i, t} = \alpha_i + \beta_{Market} R_{m, t} + \epsilon_{i, t}$$
     $$\sigma_{Idio} = \sqrt{Var(\epsilon_i)}$$
2. **分群與篩選**：特徵標準化後**跳過降維步驟**，直接作為 HDBSCAN 輸入進行分群。後續進行**同群落 × 同產業**的 EG 共整合檢定與 O-U 半衰期過濾。


## 6️⃣ 多因子分群：Trading 交易期與學術依據

1. **交易執行**：與 `HDBSCAN.py` 相同。包含雙端等市值中性配置、20 日價差滾動波動度調整 Z-Score 系統，以及虧損達 10% 的後置投資組合總體止損斷路器。
2. **文獻依據**：
   - **Fundamentals & Factors-based Pairing**：參考 [`2021-In Search of Pairs using Firm Fundamentals.pdf`](file:///d:/Unknown/Papper/Code/ref/2021-In%20Search%20of%20Pairs%20using%20Firm%20Fundamentals.pdf)。
   - **CAPM & Market Beta**：Sharpe, W. F. (1964). *"Capital Asset Prices: A Theory of Market Equilibrium under Conditions of Risk"*。
   - **Idiosyncratic Volatility Theory**：Ang, A. et al. (2006). *"The Cross-Section of Volatility and Expected Returns"*。
3. **學術邏輯**：利用 Beta、波動度、偏度、峰度以及特異波動率等因子特徵空間來進行股票聚類更具備經濟學直覺。因為具有相同貝塔、相似總波動率與特異波動率 of 股票，代表其背後承受了類似的宏觀經濟與行業微觀基本面衝擊，更容易在歷史價格偏離後實現均值復歸。


# 📈 績效分析

## 📈 六大策略最優參數回測效能對比 (實時更新)

<table style="width: 100%; border-collapse: collapse; font-family: 'Inter', 'Outfit', sans-serif; font-size: 0.52em; margin: 10px auto; text-align: center; border-radius: 8px; overflow: hidden; box-shadow: 0 4px 10px rgba(0,0,0,0.08);">
  <thead>
    <tr style="background-color: #1a365d; color: #ffffff; font-weight: 600; text-transform: uppercase;">
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">策略名稱 (Method)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最佳參數組合 (Optimal Params)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最終淨值 (Final Equity)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">年化報酬 (Ann. Return)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">最大回撤 (Max DD)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">夏普值 (Sharpe)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">RCC (%)</th>
      <th style="padding: 10px 8px; border: 1px solid #cbd5e1;">REC (%)</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color: #ffffff;">
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; font-weight: bold; text-align: left; ">經典 SSD (Basic) (CURRENT)</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">Top 5, SL: 0%, ZWin: 0</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">$8,969.99</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.43%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-20.36%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0;">-0.22</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-10.30%</td>
      <td style="padding: 8px 6px; border: 1px solid #e2e8f0; color: #e53e3e;">-0.15%</td>
    </tr>
  </tbody>
</table>

> [!NOTE]
> **RCC (Return on Capital Constraint)**: 基於回測期分配總資金 $10,000 計算。
> **REC (Return on Engaged Capital)**: 基於實際動用且對齊 Beta 避險權重的保證金資金計算。
> *資料更新時間: 當前電腦編譯實時生成。



In [4]:
#| echo: false
#| fig-align: center
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 設定標準渲染器以相容 VS Code Jupyter 與 Quarto Revealjs，解決 MIME type 報錯
pio.renderers.default = "iframe"

# 載入極輕量化、預處理好的淨值數據
df_eq = pd.read_csv("equity_curves.csv")
df_eq["Date"] = pd.to_datetime(df_eq["Date"])

# 整理為 Tidy Data 格式
df_melted = df_eq.melt(id_vars=["Date"], var_name="Strategy", value_name="Account Equity ($)")

# 縮短策略名稱為精簡中文名，保證 Legend 可讀性與視覺乾淨度
strategy_mapping = {
    "SSD_Basic": "經典 SSD",
    "SSD_OLS": "進階 SSD (OLS) 🌟",
    "Engle_Granger": "EG 共整合",
    "HDBSCAN_Handcrafted": "HDBSCAN 手動特徵",
    "HDBSCAN_Autoencoder": "HDBSCAN 自編碼器",
    "HDBSCAN_MultiFactor": "HDBSCAN 多因子"
}
df_melted["Strategy"] = df_melted["Strategy"].replace(strategy_mapping)

# 繪製極美、可互動的 Plotly 績效折線圖
fig = px.line(
    df_melted,
    x="Date",
    y="Account Equity ($)",
    color="Strategy",
    title="六大配對交易策略資產淨值曲線對比 (2000-2025)",
    color_discrete_sequence=['#4ade80', '#60a5fa', '#f87171', '#fbd38d', '#c084fc', '#2b6cb0']
)

# 設定垂直排列於右側 (orientation='v') 且調大右側 Margin 防止重疊
fig.update_layout(
    font_family="Inter, Outfit, sans-serif",
    hovermode="x unified",
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    margin=dict(l=20, r=160, t=50, b=20),
    xaxis_title="時間",
    yaxis_title="帳戶資產淨值 ($)"
)

fig.show()